In [0]:
%sql
-- Create a volume specifically for your project metadata
CREATE VOLUME IF NOT EXISTS workspace.default.checkpoints;

In [0]:
# 1. Clear the old "wrong" memory (Resetting for the fix)
dbutils.fs.rm("/Volumes/workspace/default/checkpoints/f1_schema_location", recurse=True)
dbutils.fs.rm("/Volumes/workspace/default/checkpoints/f1_incremental", recurse=True)

# 2. Run the fixed reader (Added 'inferSchema' and 'header' explicitly)
incremental_df = spark.readStream \
    .format("cloudFiles") \
    .option("cloudFiles.format", "csv") \
    .option("cloudFiles.schemaLocation", "/Volumes/workspace/default/checkpoints/f1_schema_location") \
    .option("header", "true") \
    .option("inferSchema", "true") \
    .load("/Volumes/workspace/default/raw_data/raw_data/")

# 3. Overwrite the table with the clean data
query = incremental_df.writeStream \
    .format("delta") \
    .option("checkpointLocation", "/Volumes/workspace/default/checkpoints/f1_incremental") \
    .trigger(availableNow=True) \
    .outputMode("complete") \
    .toTable("f1_processed.silver_incremental_results") # Use overwrite if needed

In [0]:
# Check the top 5 records in your new table
display(spark.read.table("f1_processed.silver_incremental_results").limit(5))

# Check the total count
print(f"Total Rows Ingested: {spark.read.table('f1_processed.silver_incremental_results').count()}")